# UNSW-NB15 LSTM Training — Single Dataset Version

Cleaned PyTorch notebook using your existing `UNSW_NB15_testing-set.parquet`.

Important fixes:
- split the raw dataset first
- fit encoding/scaling only on training data
- create train/test sequences separately
- remove duplicate training blocks
- avoid hard-coded feature count
- save all LSTM artifacts in `../deeplearn_models/`

In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

## 1. Paths

In [2]:
DATASET_PATH = "../dataset/UNSW_NB15_testing-set.parquet"
MODELS_DIR = "../deeplearn_models"

os.makedirs(MODELS_DIR, exist_ok=True)

## 2. Load dataset

In [3]:
dataset = pd.read_parquet(DATASET_PATH)

print("Dataset shape:", dataset.shape)
dataset.head()

Dataset shape: (82332, 36)


,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sload,...,trans_depth,response_body_len,ct_src_dport_ltm,ct_dst_sport_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,is_sm_ips_ports,attack_cat,label
0,0.000011,udp,-,INT,2,0,496,0,90909.09375,180363632.0,...,0,0,1,1,0,0,0,0,Normal,0
1,0.000008,udp,-,INT,2,0,1762,0,125000.00000,881000000.0,...,0,0,1,1,0,0,0,0,Normal,0
2,0.000005,udp,-,INT,2,0,1068,0,200000.00000,854400000.0,...,0,0,1,1,0,0,0,0,Normal,0
3,0.000006,udp,-,INT,2,0,900,0,166666.65625,600000000.0,...,0,0,2,1,0,0,0,0,Normal,0
4,0.000010,udp,-,INT,2,0,2126,0,100000.00000,850400000.0,...,0,0,2,1,0,0,0,0,Normal,0


## 3. Split raw rows first

In [4]:
train_df, test_df = train_test_split(
    dataset,
    test_size=0.2,
    random_state=42,
    stratify=dataset["label"]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train: (65865, 36)
Test: (16467, 36)


## 4. Separate features and labels

In [5]:
X_train = train_df.drop(columns=["label", "attack_cat"])
y_train = train_df["label"]

X_test = test_df.drop(columns=["label", "attack_cat"])
y_test = test_df["label"]

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(65865, 34) (65865,)
(16467, 34) (16467,)


## 5. One-hot encode categorical features

In [6]:
categorical_columns = ["proto", "service", "state"]

X_train = pd.get_dummies(
    X_train,
    columns=categorical_columns,
    drop_first=False
)

X_test = pd.get_dummies(
    X_test,
    columns=categorical_columns,
    drop_first=False
)

# Force test columns to match training columns
X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

print("Train encoded:", X_train.shape)
print("Test encoded:", X_test.shape)
print("Same columns:", list(X_train.columns) == list(X_test.columns))

Train encoded: (65865, 182)
Test encoded: (16467, 182)
Same columns: True


## 6. Save feature order

In [7]:
feature_columns = X_train.columns.tolist()

joblib.dump(
    feature_columns,
    os.path.join(MODELS_DIR, "lstm_feature_columns.pkl")
)

print("Input features:", len(feature_columns))

Input features: 182


## 7. Scale using training data only

In [8]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = X_train_scaled.astype(np.float32)
X_test_scaled = X_test_scaled.astype(np.float32)

joblib.dump(
    scaler,
    os.path.join(MODELS_DIR, "lstm_scaler.pkl")
)

print(X_train_scaled.shape)
print(X_test_scaled.shape)

(65865, 182)
(16467, 182)


## 8. Sequence dataset

In [9]:
sequence_length = 10

class FlowSequenceDataset(Dataset):

    def __init__(self, X, y, sequence_length=10):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(np.asarray(y), dtype=torch.float32)
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.X) - self.sequence_length + 1

    def __getitem__(self, index):
        X_sequence = self.X[
            index:index + self.sequence_length
        ]

        # Label of the last flow inside the sequence
        y_label = self.y[
            index + self.sequence_length - 1
        ]

        return X_sequence, y_label

## 9. Create sequence datasets

In [10]:
train_dataset = FlowSequenceDataset(
    X_train_scaled,
    y_train,
    sequence_length
)

test_dataset = FlowSequenceDataset(
    X_test_scaled,
    y_test,
    sequence_length
)

print("Train sequences:", len(train_dataset))
print("Test sequences:", len(test_dataset))

Train sequences: 65856
Test sequences: 16458


## 10. DataLoaders

In [11]:
batch_size = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

X_batch, y_batch = next(iter(train_loader))

print(X_batch.shape)
print(y_batch.shape)

torch.Size([128, 10, 182])
torch.Size([128])


## 11. LSTM model

In [12]:
class LSTMModel(nn.Module):

    def __init__(
        self,
        input_size,
        hidden_size=32,
        num_layers=1
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        return self.fc(out)

## 12. Device + model

In [13]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using:", device)

input_size = X_train_scaled.shape[1]

model = LSTMModel(
    input_size=input_size,
    hidden_size=32,
    num_layers=1
).to(device)

print(model)

Using: cpu
LSTMModel(
  (lstm): LSTM(182, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)


## 13. Loss + optimizer

In [14]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

## 14. Train

In [15]:
epochs = 10

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch).squeeze(1)

        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch [{epoch+1}/{epochs}], "
        f"Loss: {average_loss:.4f}"
    )

Epoch [1/10], Loss: 0.3799
Epoch [2/10], Loss: 0.2581
Epoch [3/10], Loss: 0.2381
Epoch [4/10], Loss: 0.2279
Epoch [5/10], Loss: 0.2204
Epoch [6/10], Loss: 0.2149
Epoch [7/10], Loss: 0.2099
Epoch [8/10], Loss: 0.2051
Epoch [9/10], Loss: 0.2004
Epoch [10/10], Loss: 0.1957


## 15. Evaluate

In [16]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        outputs = model(X_batch).squeeze(1)
        probabilities = torch.sigmoid(outputs)

        predictions = (probabilities >= 0.5).int()

        all_preds.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            y_batch.cpu().numpy()
        )

print("Accuracy:", accuracy_score(all_labels, all_preds))
print("Precision:", precision_score(all_labels, all_preds))
print("Recall:", recall_score(all_labels, all_preds))
print("F1:", f1_score(all_labels, all_preds))

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

Accuracy: 0.8986511119212541
Precision: 0.9415910176779742
Recall: 0.8698962701390421
F1: 0.9043248824136745

Confusion Matrix:
[[6907  489]
 [1179 7883]]


## 16. Save model + config

In [17]:
torch.save(
    model.state_dict(),
    os.path.join(MODELS_DIR, "lstm_model.pth")
)

model_config = {
    "input_size": input_size,
    "hidden_size": 32,
    "num_layers": 1,
    "sequence_length": sequence_length,
    "threshold": 0.5
}

joblib.dump(
    model_config,
    os.path.join(MODELS_DIR, "lstm_config.pkl")
)

print("Saved files:")
print(os.listdir(MODELS_DIR))

Saved files:
['live', 'lstm_config.pkl', 'lstm_feature_columns.pkl', 'lstm_model.pth', 'lstm_scaler.pkl']
